# Env38 - Load and Validate the Accepted Canonical Street Network

This professor-facing notebook uses the same tested graph adapter as Env39. It reads the accepted Env38 run through repository pointers and never downloads or rewrites network data.

## 1. Purpose
Load, inspect, and validate the accepted Env38 canonical network without regenerating it.

## 2. Research context
Env38 and Env39 share a controlled topology. Env38 retains its professor-edited geometric realization, so geometry-sensitive results may differ while identifiers and connectivity remain controlled.

## 3. Repository discovery
The discovery cell supports execution from either the repository root or the `notebooks` directory.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (start, *start.parents)
                     if (p / 'environment.yml').is_file()
                     and (p / 'src' / 'geogami_morphology').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the repository root or notebooks directory.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
ENVIRONMENT = 'env38'
print('Repository root:', PROJECT_ROOT.name)
print('REPOSITORY_DISCOVERY: PASS')

## 4. Software/version reporting
Versions are reported from the active offline environment for reproducibility.

In [ ]:
from importlib.metadata import version
import json
import platform
import networkx as nx
import pandas as pd
from IPython.display import SVG, display
from geogami_morphology.environments import get_environment
from geogami_morphology.graph import (
    build_networkx_multigraph, build_osmnx_multidigraph, directed_arc_count,
    load_analysis_graphml, load_canonical_geopackage, physical_street_count,
    resolve_canonical_run, validate_canonical_for_analysis,
    validate_graph_correspondence, validate_graphml_roundtrip,
)
from geogami_morphology.io import sha256_file
from geogami_morphology.notebook import canonical_network_svg
packages = ['osmnx', 'networkx', 'geopandas', 'shapely', 'pandas', 'numpy', 'pyproj']
display(pd.DataFrame.from_dict({'Python': platform.python_version(), **{p: version(p) for p in packages}}, orient='index', columns=['version']))
print('SOFTWARE_VERSION_REPORT: PASS')

## 5. Environment registry resolution
The environment registry is the authoritative source of Env38 input, canonical, analysis, and topology-reference paths.

In [ ]:
registry_path = PROJECT_ROOT / 'config' / 'environments.yaml'
environment = get_environment(ENVIRONMENT, registry_path=registry_path, repository_root=PROJECT_ROOT)
display(pd.Series(environment.relative_paths(), name='repository path').to_frame())
print('ENVIRONMENT_REGISTRY_RESOLUTION: PASS')

## 6. Resolving Env38 canonical latest
The active run ID is read from `data/canonical/curvilinear/latest.json`; notebook logic does not contain a fixed run ID.

In [ ]:
canonical_latest = json.loads(environment.canonical_latest.read_text(encoding='utf-8'))
selection = resolve_canonical_run(ENVIRONMENT, canonical_run='latest', latest_path=environment.canonical_latest, project_root=PROJECT_ROOT)
assert selection.run_id == canonical_latest['run_id']
print('Canonical run:', selection.run_id)
print('Canonical path:', selection.canonical_path.relative_to(PROJECT_ROOT).as_posix())
print('CANONICAL_LATEST_RESOLUTION: PASS')

## 7. Resolving Env38 analysis latest
The analysis pointer must select the same canonical identity and completed analytical run.

In [ ]:
analysis_latest_path = environment.analysis_root / 'latest.json'
analysis_latest = json.loads(analysis_latest_path.read_text(encoding='utf-8'))
assert analysis_latest['canonical_run_id'] == selection.run_id
assert analysis_latest['canonical_path'] == canonical_latest['canonical_path']
analysis_manifest_path = PROJECT_ROOT / analysis_latest['analysis_manifest_path']
analysis_run_root = analysis_manifest_path.parent
print('Analysis run:', analysis_latest['canonical_run_id'])
print('ANALYSIS_LATEST_RESOLUTION: PASS')

## 8. Canonical provenance
The canonical manifest links the editable source, frozen topology reference, configuration hashes, software, and production Git state.

In [ ]:
manifest = selection.manifest
display(pd.Series({
    'run_id': selection.run_id,
    'canonical_sha256': selection.file_sha256,
    'scientific_content_signature': selection.scientific_content_signature,
    'topology_signature': selection.topology_signature,
    'editable_source': manifest['input_editable_network']['path'],
    'topology_reference': manifest['reference_canonical']['path'],
    'production_git_commit': manifest['git_provenance']['commit_sha'],
}, name='value').to_frame())
print('CANONICAL_PROVENANCE: PASS')

## 9. Loading Env38 nodes
Canonical nodes use stable scientific `node_id` values and local Cartesian point geometry.

## 10. Loading Env38 edges
Canonical edges contain one LineString per physical street with stable `edge_id`, `u`, `v`, and `key` fields.

In [ ]:
nodes, edges = load_canonical_geopackage(selection.canonical_path)
print(f'Loaded {len(nodes)} nodes and {len(edges)} physical edges')
display(nodes.head())
display(edges.head())
print('CANONICAL_LAYERS_LOADED: PASS')

## 11. Inspecting schemas
GeoPackage feature IDs are storage details; the scientific identifiers are the explicit canonical ID fields.

In [ ]:
display(pd.DataFrame({'nodes_column': pd.Series(nodes.columns), 'nodes_dtype': pd.Series([str(x) for x in nodes.dtypes])}))
display(pd.DataFrame({'edges_column': pd.Series(edges.columns), 'edges_dtype': pd.Series([str(x) for x in edges.dtypes])}))
print('CANONICAL_SCHEMA_INSPECTION: PASS')

## 12. Local Cartesian CRS explanation
GeoGami Local Cartesian coordinates are not longitude/latitude. Distances are local units, not metres unless a justified scale transformation is established. Great-circle distance, geographic bearing, fake EPSG:4326, and geographic nearest-node functions are not used.

In [ ]:
assert not nodes.crs.is_geographic and nodes.crs == edges.crs
print('Coordinate system:', manifest['network_identity']['coordinate_system'])
print('Distance units: local units (NOT metres)')
print('LOCAL_CARTESIAN_CRS: PASS')

## 13. Canonical identity
Identity is recalculated by the tested implementation and compared with the pointer and canonical manifest.

In [ ]:
canonical = validate_canonical_for_analysis(nodes, edges)
assert sha256_file(selection.canonical_path) == selection.file_sha256
assert canonical.identity['scientific_content_signature'] == canonical_latest['scientific_content_signature']
assert canonical.identity['topology_signature'] == canonical_latest['topology_signature']
assert (canonical.identity['node_count'], canonical.identity['physical_edge_count'], canonical.identity['component_count']) == (46, 69, 1)
display(pd.Series(canonical.identity, name='value').to_frame())
print('CANONICAL_IDENTITY: 46 nodes | 69 physical edges | 1 component | PASS')

## 14. NetworkX physical graph construction/load
The scientific NetworkX `MultiGraph` contains one undirected edge per physical canonical street.

In [ ]:
G_physical = build_networkx_multigraph(canonical, selection)
components = nx.number_connected_components(G_physical)
degree_sum = sum(dict(G_physical.degree()).values())
assert type(G_physical) is nx.MultiGraph
assert (G_physical.number_of_nodes(), G_physical.number_of_edges(), components, degree_sum) == (46, 69, 1, 138)
print('NETWORKX_VALIDATION: 46 nodes | 69 physical edges | 1 component | PASS')

## 15. OSMnx MultiDiGraph construction/load
The OSMnx-compatible `MultiDiGraph` uses two reciprocal, geometry-oriented arcs for each physical street.

In [ ]:
G_osmnx = build_osmnx_multidigraph(canonical, selection)
correspondence = validate_graph_correspondence(canonical, G_physical, G_osmnx, selection)
assert type(G_osmnx) is nx.MultiDiGraph
assert (G_osmnx.number_of_nodes(), directed_arc_count(G_osmnx), physical_street_count(G_osmnx)) == (46, 138, 69)
print('OSMNX_VALIDATION: 46 nodes | 138 directed arcs | 69 physical canonical streets | PASS')

## 16. 69 physical streets versus 138 directed arcs
The 138 reciprocal arcs are an analysis representation. They must never be interpreted as 138 physical streets.

## 17. E042 reversed-coordinate-direction explanation
Canonical LineString storage may be `u` to `v` or `v` to `u`. The adapter preserves the physical geometry but creates a forward arc from `u` to `v` and an exact reverse arc.

In [ ]:
edge = canonical.edges.set_index('edge_id').loc['E042']
node_geometry = canonical.nodes.set_index('node_id').geometry
stored = tuple(edge.geometry.coords)
u_point = tuple(node_geometry.loc[str(edge.u)].coords[0])
v_point = tuple(node_geometry.loc[str(edge.v)].coords[0])
stored_direction = 'u_to_v' if (stored[0], stored[-1]) == (u_point, v_point) else 'v_to_u'
arcs = [(u, v, data) for u, v, _, data in G_osmnx.edges(keys=True, data=True) if data['canonical_edge_id'] == 'E042']
forward = next(row for row in arcs if row[2]['arc_direction'] == 'forward')
reverse = next(row for row in arcs if row[2]['arc_direction'] == 'reverse')
assert tuple(reverse[2]['geometry'].coords) == tuple(reversed(tuple(forward[2]['geometry'].coords)))
print('E042 canonical stored direction:', stored_direction)
print('E042_GRAPH_ADAPTER: PASS')

## 18. Physical-network visualization
A headless SVG is produced in memory from the canonical physical graph; no production figure is rewritten.

In [ ]:
display(SVG(canonical_network_svg(canonical.nodes, canonical.edges, title='Env38 canonical physical street network')))
print('PHYSICAL_NETWORK_VISUALIZATION: PASS')

## 19. GraphML round-trip verification
The published GraphML is loaded through OSMnx's typed loader and checked against the canonical physical graph and local CRS.

In [ ]:
analysis_graph_manifest = json.loads((analysis_run_root / 'analysis_graph_manifest.json').read_text(encoding='utf-8'))
graphml_path = PROJECT_ROOT / analysis_graph_manifest['artifacts']['graphml_path']
assert sha256_file(graphml_path) == analysis_graph_manifest['artifacts']['graphml_sha256']
G_roundtrip = load_analysis_graphml(graphml_path)
roundtrip = validate_graphml_roundtrip(canonical, G_physical, G_roundtrip, selection)
assert (G_roundtrip.number_of_nodes(), G_roundtrip.number_of_edges(), physical_street_count(G_roundtrip)) == (46, 138, 69)
print('GRAPHML_ROUNDTRIP: 46 nodes | 138 directed arcs | 69 physical IDs | PASS')

## 20. Final validation summary
This notebook validates the accepted Env38 graph without altering pointers, canonical data, GraphML, or metric artifacts.

In [ ]:
print(f'Accepted Env38 run: {selection.run_id}')
print('Coordinate system: GeoGami Local Cartesian')
print('Units: local units')
print('Internet access: NOT REQUIRED')
print('NOTEBOOK_05_EXECUTION: PASS')